[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/research/VIX_VAR_BENCHMARK.ipynb)


# VIX VAR Benchmark v1 — un VAR classique "naïf" bat-il la référence ML en walk-forward ?

**Rôle.** Complément de `VIX_VECM` : là où ce dernier n'est testable que sur les folds où
Johansen détecte une cointégration, ce notebook fournit un **benchmark économétrique
inconditionnel** — un VAR(p) estimé directement en niveaux (sans traitement de la
non-stationnarité), utilisé pour prévoir le VIX à chaque horizon et comparé, en
walk-forward strict, à la référence établie du projet (GLOBAL RandomForest :
F1_dir≈0.610±0.025).

**Volontairement naïf.** Un VAR estimé en niveaux sur des séries potentiellement I(1)
reste un choix standard en prévision pratique (contrairement à l'inférence, où la
non-stationnarité invaliderait les tests usuels) — c'est littéralement ce que ferait un
praticien pressé, sans étape de diagnostic ADF/Johansen. Ce notebook teste donc si
même cette version "sans discernement" du VAR contient un signal directionnel
comparable au ML — un plancher de comparaison, pas le résultat économétrique le plus
soigné (voir `VIX_VECM` pour la version qui traite la cointégration correctement).

**Variables** : mêmes que `VIX_VAR_MACRO`/`VIX_VECM` — VIX + pente des taux (T10Y2Y),
breakeven inflation (T10YIE), conditions financières (NFCI), taux Fed (EFFR).


In [1]:
import subprocess, sys
pkgs = ['yfinance', 'pandas_datareader', 'statsmodels', 'xlsxwriter', 'pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os, time, json, warnings, random, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pandas_datareader.data as web
from sklearn.metrics import f1_score, accuracy_score
from statsmodels.tsa.vector_ar.var_model import VAR

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_VAR_BENCHMARK'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'macro_series': {'T10Y2Y': 'T10Y2Y', 'T10YIE': 'T10YIE', 'NFCI': 'NFCI', 'EFFR': 'EFFR'},
    'start_date': '2003-01-01',
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES pour les mêmes folds
    'maxlags': 10,
    'min_train_rows': 250, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_var_benchmark_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-var-benchmark'

# Référence établie (README) pour comparaison directe — GLOBAL RandomForest h=5j N=8 SHAP SMOTE
BASELINE_F1_DIR = 0.610
BASELINE_F1_UP_FORT = 0.359
BASELINE_F1_DOWN_FORT = 0.627

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | macro: {list(CONFIG['macro_series'].keys())} | "
      f"horizons={CONFIG['horizons']} | référence à battre: F1_dir={BASELINE_F1_DIR}")


VIX_VAR_BENCHMARK v1 | macro: ['T10Y2Y', 'T10YIE', 'NFCI', 'EFFR'] | horizons=[1, 2, 3, 5, 7, 10] | référence à battre: F1_dir=0.61


In [3]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES) — pour retrouver
# exactement le même VIX_COL, les mêmes dates et les mêmes coupures de fold que la
# référence ML établie du projet (comparaison à isométrie de méthode).
# ============================================================
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
VIX_COL = meta['vix_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


[PULL OK] Dataset récupéré depuis 'results/vix-final-features'
Dataset: (6908, 1300) | VIX=IDX_VIX | source: 2000-01-03 → 2026-07-21
  Fold 1: train → 2010-08-16 | test 2010-08-17 → 2013-10-21
  Fold 2: train → 2013-10-21 | test 2013-10-22 → 2016-12-28
  Fold 3: train → 2016-12-28 | test 2016-12-29 → 2020-03-06
  Fold 4: train → 2020-03-06 | test 2020-03-09 → 2023-05-12
  Fold 5: train → 2023-05-12 | test 2023-05-15 → 2026-07-21


In [4]:
# ============================================================
# SYSTÈME VAR/VECM : VIX (issu du dataset partagé, pour cohérence avec la
# référence ML) + variables macro FRED, alignées sur le même index.
# ============================================================
system_df = pd.DataFrame(index=df_features.index)
system_df['VIX'] = df_features[VIX_COL]

for name, sid in CONFIG['macro_series'].items():
    try:
        s = web.DataReader(sid, 'fred', CONFIG['start_date']).squeeze()
        system_df[name] = s.reindex(system_df.index, method='ffill')
    except Exception as e:
        print(f"  [WARN] FRED {sid}: {str(e)[:100]}")

VAR_COLS = list(system_df.columns)
VIX_IDX = VAR_COLS.index('VIX')
system_df = system_df.ffill().dropna()
print(f"Système: {system_df.shape} | colonnes: {VAR_COLS} | {system_df.index.min().date()} → "
      f"{system_df.index.max().date()}")


Système: (6127, 5) | colonnes: ['VIX', 'T10Y2Y', 'T10YIE', 'NFCI', 'EFFR'] | 2003-01-03 → 2026-07-21


In [5]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def classify_return(r, reg, thr):
    q25, q75 = thr.get(reg, (0, 0))
    if r < q25: return 0
    if r < 0:   return 1
    if r < q75: return 2
    return 3

print("Helpers OK (build_target, metrics, classify_return)")


Helpers OK (build_target, metrics, classify_return)


In [6]:
# ============================================================
# MOTEUR : par fold walk-forward, VAR(p) en niveaux (train) -> prévision glissante
# à chaque date du test via VARResults.forecast() -> classe -> métriques.
# ============================================================
rows = []
t0 = time.time()

for k in range(CONFIG['n_wf_folds']):
    cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
    cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
    train = system_df.loc[system_df.index < cut_date, VAR_COLS].dropna()
    if len(train) < CONFIG['min_train_rows']:
        print(f"Fold {k+1}: train trop court ({len(train)}) — ignoré.")
        continue

    sel = VAR(train).select_order(CONFIG['maxlags'])
    p = sel.aic if sel.aic and sel.aic > 0 else 1
    var_res = VAR(train).fit(p)
    stable = var_res.is_stable()
    print(f"\nFold {k+1}: train={len(train)}j | p={p} | {'stable' if stable else 'INSTABLE'}")

    for h in CONFIG['horizons']:
        target, reg_r, thr = build_target(system_df['VIX'], h, cut)
        te_idx = target.index[(target.index >= cut_date) & (target.index <= nxt_date)]
        if len(te_idx) < CONFIG['min_test_rows']:
            continue

        y_true, y_pred = [], []
        for t in te_idx:
            pos = system_df.index.get_loc(t)
            if pos < p - 1:
                continue
            y_hist = system_df[VAR_COLS].iloc[pos - p + 1: pos + 1].values
            fc = var_res.forecast(y_hist, steps=h)[-1]
            vix_fc = fc[VIX_IDX]; vix_now = system_df['VIX'].iloc[pos]
            ret_hat = vix_fc / vix_now - 1
            pred = classify_return(ret_hat, reg_r[t], thr)
            y_true.append(int(target.loc[t])); y_pred.append(pred)

        if len(y_true) < CONFIG['min_test_rows']:
            continue
        met = metrics(np.array(y_true), np.array(y_pred))
        rows.append({'fold': k + 1, 'horizon': h, 'lag_p': p, 'stable': stable,
                     'n_test': len(y_true), 'test_start': str(cut_date.date()),
                     'test_end': str(nxt_date.date()), **met})
        print(f"  h={h:2d}j n={len(y_true):4d} F1_dir={met['F1_dir']:.3f} "
              f"F1_UP_FORT={met['F1_UP_FORT']} F1_DOWN_FORT={met['F1_DOWN_FORT']}")

df_var_bench = pd.DataFrame(rows)
df_var_bench.to_csv(RESULTS_CSV, index=False)
print(f"\n[TERMINÉ] {len(df_var_bench)} lignes (fold × horizon) en {(time.time()-t0)/60:.1f}min "
      f"-> {RESULTS_CSV}")


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Fold 1: train=1982j | p=10 | stable
  h= 1j n= 758 F1_dir=0.562 F1_UP_FORT=0.2063 F1_DOWN_FORT=0.2088
  h= 2j n= 794 F1_dir=0.596 F1_UP_FORT=0.2509 F1_DOWN_FORT=0.2047
  h= 3j n= 813 F1_dir=0.609 F1_UP_FORT=0.2617 F1_DOWN_FORT=0.2146
  h= 5j n= 807 F1_dir=0.632 F1_UP_FORT=0.3772 F1_DOWN_FORT=0.2671
  h= 7j n= 804 F1_dir=0.630 F1_UP_FORT=0.4405 F1_DOWN_FORT=0.306
  h=10j n= 819 F1_dir=0.638 F1_UP_FORT=0.5474 F1_DOWN_FORT=0.2437

Fold 2: train=2811j | p=10 | stable


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


  h= 1j n= 771 F1_dir=0.466 F1_UP_FORT=0.4706 F1_DOWN_FORT=0.0395
  h= 2j n= 806 F1_dir=0.484 F1_UP_FORT=0.588 F1_DOWN_FORT=0.0449
  h= 3j n= 811 F1_dir=0.489 F1_UP_FORT=0.6025 F1_DOWN_FORT=0.0504
  h= 5j n= 816 F1_dir=0.481 F1_UP_FORT=0.6188 F1_DOWN_FORT=0.0676
  h= 7j n= 817 F1_dir=0.459 F1_UP_FORT=0.6614 F1_DOWN_FORT=0.0278
  h=10j n= 811 F1_dir=0.441 F1_UP_FORT=0.6838 F1_DOWN_FORT=0.0143


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Fold 3: train=3640j | p=10 | stable
  h= 1j n= 767 F1_dir=0.570 F1_UP_FORT=0.1391 F1_DOWN_FORT=0.1502
  h= 2j n= 802 F1_dir=0.580 F1_UP_FORT=0.1791 F1_DOWN_FORT=0.1678
  h= 3j n= 808 F1_dir=0.590 F1_UP_FORT=0.2051 F1_DOWN_FORT=0.2424
  h= 5j n= 818 F1_dir=0.631 F1_UP_FORT=0.2517 F1_DOWN_FORT=0.3303
  h= 7j n= 814 F1_dir=0.629 F1_UP_FORT=0.2088 F1_DOWN_FORT=0.3835
  h=10j n= 819 F1_dir=0.629 F1_UP_FORT=0.2529 F1_DOWN_FORT=0.3559


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Fold 4: train=4469j | p=10 | stable
  h= 1j n= 779 F1_dir=0.496 F1_UP_FORT=0.0543 F1_DOWN_FORT=0.2308
  h= 2j n= 807 F1_dir=0.490 F1_UP_FORT=0.0179 F1_DOWN_FORT=0.2243
  h= 3j n= 809 F1_dir=0.451 F1_UP_FORT=0.0 F1_DOWN_FORT=0.3401
  h= 5j n= 813 F1_dir=0.426 F1_UP_FORT=0.0 F1_DOWN_FORT=0.4615
  h= 7j n= 812 F1_dir=0.432 F1_UP_FORT=0.0 F1_DOWN_FORT=0.4634
  h=10j n= 818 F1_dir=0.388 F1_UP_FORT=0.0 F1_DOWN_FORT=0.4734


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Fold 5: train=5298j | p=10 | stable
  h= 1j n= 763 F1_dir=0.557 F1_UP_FORT=0.0316 F1_DOWN_FORT=0.0744
  h= 2j n= 801 F1_dir=0.550 F1_UP_FORT=0.0197 F1_DOWN_FORT=0.0796
  h= 3j n= 802 F1_dir=0.565 F1_UP_FORT=0.0195 F1_DOWN_FORT=0.131
  h= 5j n= 811 F1_dir=0.577 F1_UP_FORT=0.0097 F1_DOWN_FORT=0.1429
  h= 7j n= 809 F1_dir=0.590 F1_UP_FORT=0.0 F1_DOWN_FORT=0.1695
  h=10j n= 805 F1_dir=0.565 F1_UP_FORT=0.0 F1_DOWN_FORT=0.1525

[TERMINÉ] 30 lignes (fold × horizon) en 0.6min -> vix_var_benchmark_results.csv


In [7]:
# ============================================================
# SYNTHÈSE : VAR classique (niveaux, naïf) vs référence ML établie
# ============================================================
if len(df_var_bench):
    by_h = df_var_bench.groupby('horizon')[['F1_dir', 'F1_UP_FORT', 'F1_DOWN_FORT']].mean().round(4)
    print("### Moyenne par horizon (tous folds) ###")
    print(by_h.to_string())

    overall_f1 = df_var_bench['F1_dir'].mean()
    n_unstable = int((~df_var_bench['stable']).sum())
    print(f"\nF1_dir moyen (tous folds/horizons): {overall_f1:.4f} (référence ML: {BASELINE_F1_DIR})")
    if n_unstable:
        print(f"[NOTE] {n_unstable}/{len(df_var_bench)} lignes proviennent d'un VAR jugé "
              f"instable sur son fold (racines dans/proches du cercle unité) — prévisions "
              f"potentiellement peu fiables sur ces lignes.")

    if overall_f1 >= BASELINE_F1_DIR:
        print("\n[VERDICT] Le VAR classique (niveaux, sans traitement de la non-stationnarité) "
              "égale ou dépasse la référence ML — résultat surprenant à re-vérifier "
              "(cf. VIX_VECM pour la version qui traite la cointégration proprement).")
    else:
        print(f"\n[VERDICT] Le VAR classique reste en-dessous de la référence ML (delta="
              f"{overall_f1 - BASELINE_F1_DIR:+.4f}) — cohérent avec le reste du projet : "
              "un modèle linéaire à 5 variables, même bien spécifié, ne rivalise pas avec "
              "un ensemble ML multi-features sur ce problème.")

    try:
        with pd.ExcelWriter('VIX_VAR_BENCHMARK_report.xlsx', engine='xlsxwriter') as w:
            df_var_bench.to_excel(w, 'Detail', index=False)
            by_h.reset_index().to_excel(w, 'Par_horizon', index=False)
            pd.DataFrame([{'F1_dir_moyen': overall_f1, 'F1_dir_reference_ML': BASELINE_F1_DIR,
                            'delta': overall_f1 - BASELINE_F1_DIR,
                            'lignes_instables': n_unstable}]).to_excel(w, 'Meta', index=False)
        print("\n[SAVE] VIX_VAR_BENCHMARK_report.xlsx")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat.")


### Moyenne par horizon (tous folds) ###
         F1_dir  F1_UP_FORT  F1_DOWN_FORT
horizon                                  
1        0.5302      0.1804        0.1407
2        0.5399      0.2111        0.1443
3        0.5409      0.2178        0.1957
5        0.5495      0.2515        0.2539
7        0.5483      0.2621        0.2700
10       0.5321      0.2968        0.2480

F1_dir moyen (tous folds/horizons): 0.5401 (référence ML: 0.61)

[VERDICT] Le VAR classique reste en-dessous de la référence ML (delta=-0.0699) — cohérent avec le reste du projet : un modèle linéaire à 5 variables, même bien spécifié, ne rivalise pas avec un ensemble ML multi-features sur ce problème.

[SAVE] VIX_VAR_BENCHMARK_report.xlsx


In [8]:
# ============================================================
# PUSH DU RAPPORT SUR results/vix-var-benchmark
# ============================================================
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_var_bench_push"

def push_report():
    files = [f for f in [RESULTS_CSV, 'VIX_VAR_BENCHMARK_report.xlsx'] if os.path.exists(f)]
    if not GITHUB_TOKEN or not files:
        print("[SKIP] Pas de token ou rien à pousser.")
        return
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0:
            print(f"[WARN] clone: {clone.stderr[-300:]}"); return
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        for f in files:
            subprocess.run(["cp", f, f"{_PUSH_WORKDIR}/{f}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email",
                         "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name",
                         "VIX VAR Benchmark Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add"] + files, check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport VAR Benchmark — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] {files} sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_report()


[PUSH OK] ['vix_var_benchmark_results.csv', 'VIX_VAR_BENCHMARK_report.xlsx'] sur 'results/vix-var-benchmark'
